# bob, explained - Episode 2: JTAG

the TAP, the instruction register, the boundary ring

Run `!pip install manim` and `from manim import *` once first, then this cell. Start at `-ql`; the class names listed at the top of the cell render a single section.


In [ ]:
%%manim -qm Ep02JTAG
# =============================================================================
#  bob, explained - EPISODE 2: JTAG
#  the TAP, the instruction register, the boundary ring
#
#  GENERATED by docs/manim/build.py from docs/manim/parts/. Do not edit here.
#
#  Prerequisite (once per notebook, in a cell of its own):
#      !pip install manim
#      from manim import *
#
#  Quality on the magic line above:  -ql draft   -qm medium   -qh 1080p60
#
#  Render one section instead of the whole episode by putting any of these
#  class names on the magic line:
#      E02S1Why
#      E02S2Tap
#      E02S3Timing
#      E02S4Ir
#      E02S5Status
#      E02S6Boundary
#      E02S7Files
# =============================================================================

# =============================================================================
#  shared prelude - palette, helpers and the BobScene base class.
#  docs/manim/build.py pastes this into the top of every episode cell.
# =============================================================================

from manim import *
import numpy as np

# ---------------------------------------------------------------- palette ----
BG    = "#11121a"
INK   = "#e8e8ea"
DIM   = "#8b93a7"
C_PY  = "#7aa2f7"   # blue    - Python / tools / the device description
C_VPR = "#f7768e"   # red     - VPR / external tools
C_RTL = "#9ece6a"   # green   - hardware, Verilog, things on the die
C_BIT = "#e0af68"   # amber   - configuration bits, FASM, the bitstream
C_GRF = "#bb9af7"   # purple  - graphs, JTAG, protocol
C_ERR = "#ff7a93"   # pink    - bugs, refusals, errors
MONO  = "monospace"


# ---------------------------------------------------------------- helpers ----
def mono(s, size=22, color=INK):
    """One line of monospace text (Pango crashes on '', so blanks become ' ')."""
    return Text(s if s else " ", font=MONO, font_size=size, color=color)


def code_block(lines, size=20, color=INK):
    g = VGroup(*[mono(l, size, color) for l in lines])
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.14)
    return g


def panel(mob, color=DIM, pad=0.32, fill=0.06):
    r = SurroundingRectangle(mob, color=color, buff=pad)
    r.set_fill(color, opacity=fill)
    return VGroup(r, mob)


def chip(label, color, w=2.6, h=0.95, size=22, weight="NORMAL"):
    box = RoundedRectangle(width=w, height=h, corner_radius=0.14,
                           color=color, stroke_width=3)
    box.set_fill(color, opacity=0.12)
    txt = Text(label, font_size=size, color=INK, weight=weight, line_spacing=0.75)
    if txt.width > w - 0.3:
        txt.scale_to_fit_width(w - 0.3)
    if txt.height > h - 0.2:
        txt.scale_to_fit_height(h - 0.2)
    return VGroup(box, txt.move_to(box.get_center()))


def arrow(a, b, color=DIM, buff=0.15, sw=3):
    return Arrow(a, b, buff=buff, color=color, stroke_width=sw,
                 max_tip_length_to_length_ratio=0.18)


def mux_symbol(color=C_RTL, h=1.9, w=0.8):
    """Classic trapezoid multiplexer symbol."""
    p = Polygon([-w / 2,  h / 2, 0], [w / 2,  h / 2 - 0.3, 0],
                [ w / 2, -h / 2 + 0.3, 0], [-w / 2, -h / 2, 0],
                color=color, stroke_width=3)
    p.set_fill(color, opacity=0.14)
    return p


def bitcells(n, size=0.3, on=(), color=C_BIT, off_color=DIM):
    """A strip of n little squares; indices in `on` are filled."""
    g = VGroup()
    for i in range(n):
        s = Square(size, color=off_color, stroke_width=1.6)
        if i in on:
            s.set_stroke(color).set_fill(color, opacity=0.85)
        g.add(s)
    g.arrange(RIGHT, buff=0.035)
    return g


def fieldbar(fields, total_w=11.0, h=0.62, size=15):
    """
    fields: [(label, nbits, color), ...] -> one horizontal bar split to scale,
    each slice labelled above and its bit range below. Returns VGroup(bar, labels, ranges).
    """
    nbits = sum(f[1] for f in fields)
    bar, labs, rngs = VGroup(), VGroup(), VGroup()
    x, lo = -total_w / 2, 0
    for label, n, col in fields:
        w = max(total_w * n / nbits, 0.34)
        r = Rectangle(width=w, height=h, color=col, stroke_width=2)
        r.set_fill(col, opacity=0.28).move_to(np.array([x + w / 2, 0, 0]))
        bar.add(r)
        t = Text(label, font_size=size, color=col)
        if t.width > w * 1.9:
            t.scale_to_fit_width(max(w * 1.9, 0.5))
        t.next_to(r, UP, buff=0.14)
        labs.add(t)
        rt = mono(f"{lo}" if n == 1 else f"{lo}..{lo + n - 1}", size - 2, DIM)
        rt.next_to(r, DOWN, buff=0.12)
        if rt.width > w * 1.9:
            rt.scale_to_fit_width(max(w * 1.9, 0.5))
        rngs.add(rt)
        x += w
        lo += n
    return VGroup(bar, labs, rngs)


def filecard(path, role, color):
    """A small card naming a repo file and what it is."""
    t = mono(path, 17, color)
    r = Text(role, font_size=14, color=DIM)
    g = VGroup(t, r).arrange(DOWN, aligned_edge=LEFT, buff=0.08)
    box = SurroundingRectangle(g, color=color, buff=0.16)
    box.set_fill(color, opacity=0.07)
    return VGroup(box, g)


def mid(a, b):
    """midpoint, defined here so nothing depends on manim exporting space_ops."""
    return (a + b) / 2


def clear_all(sc, run_time=0.6):
    if sc.mobjects:
        sc.play(*[FadeOut(m) for m in sc.mobjects], run_time=run_time)


class BobScene(Scene):
    def setup(self):
        self.camera.background_color = BG

    def heading(self, text, kicker=None):
        t = Text(text, font_size=32, color=INK, weight="BOLD")
        t.to_corner(UL).shift(DOWN * 0.1)
        rule = Line(LEFT * 6.6, RIGHT * 6.6, color=DIM, stroke_width=1.5)
        rule.next_to(t, DOWN, buff=0.2).align_to(t, LEFT)
        g = VGroup(t, rule)
        self.play(FadeIn(t, shift=RIGHT * 0.3), Create(rule), run_time=0.7)
        if kicker:
            k = Text(kicker, font_size=19, color=DIM)
            if k.width > 13.0:
                k.scale_to_fit_width(13.0)
            k.next_to(rule, DOWN, buff=0.16).align_to(t, LEFT)
            g.add(k)
            self.play(FadeIn(k), run_time=0.4)
        return g

    def titlecard(self, number, title, subtitle):
        n = Text(number, font_size=26, color=C_BIT, weight="BOLD")
        t = Text(title, font_size=60, color=INK, weight="BOLD")
        s = Text(subtitle, font_size=26, color=DIM)
        if t.width > 12.5:
            t.scale_to_fit_width(12.5)
        if s.width > 12.5:
            s.scale_to_fit_width(12.5)
        g = VGroup(n, t, s).arrange(DOWN, buff=0.4)
        self.play(FadeIn(n), run_time=0.4)
        self.play(Write(t), run_time=1.1)
        self.play(FadeIn(s, shift=UP * 0.2), run_time=0.7)
        self.wait(1.6)
        self.play(FadeOut(g), run_time=0.6)

    def files_used(self, inputs, generated, verified):
        """Closing card: what this episode's topic is built from and checked by."""
        self.heading("Files", "what this part is written in, what is generated, and what proves it")
        cols = []
        for title, items, col in (("written by hand", inputs, C_RTL),
                                  ("generated", generated, C_PY),
                                  ("verified by", verified, C_BIT)):
            head = Text(title, font_size=21, color=col, weight="BOLD")
            cards = VGroup(*[filecard(p, r, col) for p, r in items])
            cards.arrange(DOWN, aligned_edge=LEFT, buff=0.18)
            g = VGroup(head, cards).arrange(DOWN, aligned_edge=LEFT, buff=0.28)
            cols.append(g)
        row = VGroup(*cols).arrange(RIGHT, buff=0.7, aligned_edge=UP)
        if row.width > 13.2:
            row.scale_to_fit_width(13.2)
        row.next_to(self.mobjects[1], DOWN, buff=0.55).set_x(0)
        for c in cols:
            self.play(FadeIn(c, shift=UP * 0.2), run_time=0.7)
        self.wait(2.4)

# =============================================================================
#  EPISODE 2 - JTAG: the TAP, the instruction register, the boundary ring
# =============================================================================

def s1_why(sc):
    sc.heading("Four wires and a state machine",
               "everything that ever reaches bob - configuration, readback, test - goes through here")

    mac = chip("Mac\nhost/*.py", C_PY, 2.4, 1.3, 21).move_to(np.array([-5.0, 1.4, 0]))
    pico = chip("Pico\nDirtyJTAG", C_VPR, 2.4, 1.3, 21).move_to(np.array([-1.6, 1.4, 0]))
    pl = chip("XC7Z020 PL\nbob", C_RTL, 2.8, 1.3, 21).move_to(np.array([2.6, 1.4, 0]))
    sc.play(FadeIn(mac), run_time=0.4)
    sc.play(GrowArrow(arrow(mac.get_right(), pico.get_left(), DIM, 0.08)),
            FadeIn(pico), run_time=0.5)
    usb = mono("USB", 15, DIM).move_to(mid(mac.get_right(), pico.get_left()) + UP * 0.25)
    sc.play(FadeIn(usb), run_time=0.3)

    wires = VGroup()
    names = [("TCK", "the only clock in the whole chip"),
             ("TMS", "walks the state machine"),
             ("TDI", "data in"),
             ("TDO", "data out")]
    for k, (n, why) in enumerate(names):
        y = 1.4 + 0.45 - k * 0.3
        ln = Line(pico.get_right() + RIGHT * 0.05, np.array([1.2, y, 0]),
                  color=C_GRF, stroke_width=2.2).set_y(y)
        ln.put_start_and_end_on(np.array([-0.35, y, 0]), np.array([1.2, y, 0]))
        t = mono(n, 15, C_GRF).next_to(ln, RIGHT, buff=0.08)
        wires.add(VGroup(ln, t))
    sc.play(FadeIn(pl), LaggedStart(*[Create(w) for w in wires], lag_ratio=0.15),
            run_time=1.2)

    tbl = VGroup()
    for n, why in names:
        tbl.add(VGroup(mono(n, 19, C_GRF), Text(why, font_size=17, color=DIM))
                .arrange(RIGHT, buff=0.3, aligned_edge=DOWN))
    tbl.arrange(DOWN, aligned_edge=LEFT, buff=0.22)
    tbl.next_to(pl, DOWN, buff=1.0).set_x(-0.6)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.2) for r in tbl], lag_ratio=0.2),
            run_time=1.4)

    big = Text("TCK clocks the entire chip. There is no free-running core clock for a "
               "scan to race against - one clock domain, no CDC to get wrong.",
               font_size=20, color=C_BIT)
    big.scale_to_fit_width(12.8).to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(big), run_time=0.9)
    sc.wait(2.2)


def s2_tap(sc):
    sc.heading("The TAP state machine (IEEE 1149.1)",
               "sixteen states, and TMS alone decides which one you are in")

    def st(label, x, y, col=DIM):
        b = RoundedRectangle(width=1.85, height=0.46, corner_radius=0.1,
                             color=col, stroke_width=2).set_fill(col, opacity=0.10)
        t = Text(label, font_size=15, color=INK)
        if t.width > 1.7:
            t.scale_to_fit_width(1.7)
        return VGroup(b.move_to(np.array([x, y, 0])), t.move_to(np.array([x, y, 0])))

    tlr = st("Test-Logic-Reset", -4.6, 2.35, C_ERR)
    rti = st("Run-Test/Idle", -4.6, 1.35, C_BIT)
    sc.play(FadeIn(tlr), FadeIn(rti),
            GrowArrow(arrow(tlr.get_bottom(), rti.get_top(), DIM, 0.05)), run_time=0.7)

    ys = [1.35, 0.55, -0.25, -1.05, -1.85, -2.65, -3.3]
    dr_names = ["Select-DR", "Capture-DR", "Shift-DR", "Exit1-DR",
                "Pause-DR", "Exit2-DR", "Update-DR"]
    ir_names = [n.replace("DR", "IR") for n in dr_names]
    dr = VGroup(*[st(n, -1.2, y, C_GRF) for n, y in zip(dr_names, ys)])
    ir = VGroup(*[st(n, 2.9, y, C_PY) for n, y in zip(ir_names, ys)])

    sc.play(LaggedStart(*[FadeIn(s) for s in dr], lag_ratio=0.1), run_time=1.0)
    sc.play(LaggedStart(*[FadeIn(s) for s in ir], lag_ratio=0.1), run_time=1.0)
    e1 = VGroup(*[arrow(dr[i].get_bottom(), dr[i + 1].get_top(), DIM, 0.04, 2)
                  for i in range(6)])
    e2 = VGroup(*[arrow(ir[i].get_bottom(), ir[i + 1].get_top(), DIM, 0.04, 2)
                  for i in range(6)])
    sc.play(Create(e1), Create(e2),
            GrowArrow(arrow(rti.get_right(), dr[0].get_left(), DIM, 0.05, 2)),
            GrowArrow(arrow(dr[0].get_right(), ir[0].get_left(), DIM, 0.05, 2)),
            run_time=1.0)

    lg = code_block(["TMS = 1 moves you along;  TMS = 0 holds or drops in",
                     "five 1s from anywhere always land in Test-Logic-Reset"], 18, DIM)
    lg.to_edge(DOWN, buff=0.28)
    sc.play(FadeIn(lg), run_time=0.7)
    sc.wait(1.0)

    walk = [rti, dr[0], dr[1], dr[2], dr[3], dr[6]]
    cap = Text("one DR scan", font_size=22, color=C_BIT).move_to(np.array([5.6, 2.2, 0]))
    sc.play(FadeIn(cap), run_time=0.4)
    notes = ["idle", "select", "CAPTURE: load the register",
             "SHIFT: one bit per TCK", "leave", "UPDATE: commit"]
    prev = None
    for s, n in zip(walk, notes):
        box = SurroundingRectangle(s, color=C_BIT, buff=0.06)
        t = Text(n, font_size=18, color=C_BIT).move_to(np.array([5.6, s.get_y(), 0]))
        if t.width > 3.2:
            t.scale_to_fit_width(3.2)
        sc.play(Create(box), FadeIn(t), run_time=0.45)
        if prev:
            sc.play(FadeOut(prev[0]), FadeOut(prev[1]), run_time=0.18)
        prev = (box, t)
        sc.wait(0.45)
    sc.wait(1.6)


def s3_timing(sc):
    sc.heading("The timing contract",
               "proven on hardware and never changed since the original bob - one edge wrong and nothing answers")

    n, w, h, x0, y0 = 6, 0.95, 0.55, -5.4, 1.5
    pts = [np.array([x0, y0, 0])]
    for i in range(n):
        x = x0 + i * w
        pts += [np.array([x + w / 2, y0, 0]), np.array([x + w / 2, y0 + h, 0]),
                np.array([x + w, y0 + h, 0]), np.array([x + w, y0, 0])]
    clk = VMobject(color=C_GRF, stroke_width=3)
    clk.set_points_as_corners(pts)
    lab = mono("TCK", 20, C_GRF).next_to(clk, LEFT, buff=0.25).set_y(y0 + h / 2)
    sc.play(Create(clk), FadeIn(lab), run_time=1.2)

    rise = VGroup(*[DashedLine(np.array([x0 + (i + 0.5) * w, y0 + h + 0.35, 0]),
                               np.array([x0 + (i + 0.5) * w, -2.4, 0]),
                               color=C_BIT, stroke_width=1.6, dash_length=0.08)
                    for i in range(n)])
    fall = VGroup(*[DashedLine(np.array([x0 + (i + 1) * w, y0 + h + 0.35, 0]),
                               np.array([x0 + (i + 1) * w, -2.4, 0]),
                               color=C_VPR, stroke_width=1.6, dash_length=0.08)
                    for i in range(n - 1)])
    sc.play(Create(rise), run_time=0.7)
    r1 = Text("RISING edge", font_size=20, color=C_BIT)
    r1.next_to(clk, UP, buff=0.5).set_x(-2.0)
    sc.play(FadeIn(r1), run_time=0.4)
    rl = code_block([
        "TMS is sampled here  -> the state machine advances",
        "TDI is sampled here  -> one bit enters the shift register",
    ], 19, C_BIT)
    rl.next_to(clk, DOWN, buff=0.85).set_x(0)
    sc.play(FadeIn(rl), run_time=0.8)
    sc.wait(0.8)

    sc.play(Create(fall), run_time=0.7)
    f1 = Text("FALLING edge", font_size=20, color=C_VPR)
    f1.next_to(clk, UP, buff=0.5).set_x(2.6)
    sc.play(FadeIn(f1), run_time=0.4)
    fl = code_block([
        "TDO is launched here          - half a cycle of margin",
        "IR / DR update latches fire   - the boundary moves as a unit",
        "the configuration memory takes its frame",
    ], 19, C_VPR)
    fl.next_to(rl, DOWN, buff=0.45).align_to(rl, LEFT)
    sc.play(FadeIn(fl), run_time=0.9)

    lsb = Text("Every shift register is LSB-first: chain bit k is the k-th bit shifted in.",
               font_size=21, color=INK)
    lsb.to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(lsb), run_time=0.7)
    sc.wait(2.2)


def s4_ir(sc):
    sc.heading("A 6-bit instruction register",
               "AMD 7-series codes where they exist, so urjtag's discovery maps bob with no database entry")

    rows = [
        ("001001", "IDCODE",     "0xFBEEF093 - which build is in the PL", C_GRF),
        ("001000", "USERCODE",   "the milestone number", C_GRF),
        ("000001", "SAMPLE",     "watch the pins, touch nothing", C_PY),
        ("100110", "EXTEST",     "the boundary drives the pads", C_PY),
        ("000111", "INTEST",     "the boundary drives the FABRIC   (private)", C_PY),
        ("000010", "USER1",      "ce / sr / cin / step + 16 CLB outputs", C_BIT),
        ("000011", "USER2 = CFG_CTRL", "chain CRC and status, behind a write key", C_BIT),
        ("100010", "USER3 = CAPTURE",  "a snapshot of all 100 CLB outputs", C_BIT),
        ("100011", "USER4",      "BRAM contents / drive / SELECT", C_BIT),
        ("101000", "DSP",        "DSP drive word and both P values   (private)", C_BIT),
        ("000101", "CFG_IN",     "frame packets in", C_VPR),
        ("000100", "CFG_OUT",    "frame packets / readback out", C_VPR),
        ("110101", "CHAIN_IN",   "the whole memory as one scan   (private)", C_VPR),
        ("110100", "CHAIN_OUT",  "the same, read back   (private)", C_VPR),
        ("001011", "JPROGRAM",   "clear the configuration, drop DONE", C_ERR),
        ("001100", "JSTART",     "step the startup sequence", C_ERR),
        ("111111", "BYPASS",     "one flip-flop", DIM),
    ]
    g = VGroup()
    for code, name, why, col in rows:
        c = mono(code, 16, col)
        n = mono(name, 16, INK)
        n.align_to(c, DOWN)
        w = Text(why, font_size=14, color=DIM)
        row = VGroup(c, n, w).arrange(RIGHT, buff=0.3, aligned_edge=DOWN)
        g.add(row)
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 0.9)
        r[2].align_to(g[0][2], LEFT).shift(RIGHT * 3.4)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.12)
    g.scale_to_fit_height(5.3).next_to(sc.mobjects[1], DOWN, buff=0.5).set_x(-0.6)
    sc.play(LaggedStart(*[FadeIn(r, shift=RIGHT * 0.15) for r in g], lag_ratio=0.07),
            run_time=3.0)
    sc.wait(1.4)

    note = Text("unknown codes fall back to BYPASS - never to something that writes",
                font_size=19, color=DIM)
    note.to_edge(DOWN, buff=0.25)
    sc.play(FadeIn(note), run_time=0.7)
    sc.wait(1.8)


def s5_status(sc):
    sc.heading("Every IR scan is also a status read",
               "Capture-IR loads status instead of zeros - in the spirit of 7-series parts")

    bits = [("DONE", C_RTL), ("INIT_B", C_GRF), ("COMMITTED", C_BIT),
            ("CRC_ERR", C_ERR), ("0", DIM), ("1", DIM)]
    cells = VGroup()
    for name, col in bits:
        b = Square(1.0, color=col, stroke_width=2.5).set_fill(col, opacity=0.18)
        t = Text(name, font_size=15, color=col)
        if t.width > 0.9:
            t.scale_to_fit_width(0.9)
        cells.add(VGroup(b, t.move_to(b)))
    cells.arrange(RIGHT, buff=0.12).shift(UP * 1.2)
    idx = VGroup(*[mono(str(5 - i), 16, DIM).next_to(c, UP, buff=0.14)
                   for i, c in enumerate(cells)])
    sc.play(LaggedStart(*[FadeIn(c, scale=0.7) for c in cells], lag_ratio=0.12),
            FadeIn(idx), run_time=1.3)

    exp = code_block([
        "DONE       startup finished, LD3 is lit",
        "INIT_B     no CRC / length / packet error",
        "COMMITTED  a good load is in the memory",
        "CRC_ERR    the last load was rejected",
        "the two low bits are 01, which IEEE 1149.1 requires",
    ], 20, INK)
    exp[4].set_color(DIM)
    exp.next_to(cells, DOWN, buff=0.7).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l, shift=RIGHT * 0.15) for l in exp], lag_ratio=0.2),
            run_time=1.8)

    why = Text("so a host can ask 'are you alive, and did my bitstream take?' "
               "without selecting any data register at all",
               font_size=20, color=C_BIT)
    why.scale_to_fit_width(12.6).to_edge(DOWN, buff=0.4)
    sc.play(FadeIn(why), run_time=0.8)
    sc.wait(2.2)


def s6_boundary(sc):
    sc.heading("The boundary ring", "hw/src/core/bsc_cell.v - two flip-flops per cell, "
                                    "and that split is the whole point")

    dpin = mono("data_in", 17, DIM).move_to(np.array([-5.6, 1.3, 0]))
    dout = mono("data_out", 17, DIM).move_to(np.array([3.6, 1.3, 0]))
    wire = Line(np.array([-4.8, 1.3, 0]), np.array([2.2, 1.3, 0]),
                color=DIM, stroke_width=2)
    m = mux_symbol(C_RTL, 1.1, 0.5).move_to(np.array([2.6, 1.3, 0]))
    sc.play(FadeIn(dpin), Create(wire), Create(m), FadeIn(dout), run_time=0.9)

    cap = chip("capture_reg", C_GRF, 2.4, 0.8, 19).move_to(np.array([-2.0, -0.4, 0]))
    upd = chip("update_reg", C_BIT, 2.4, 0.8, 19).move_to(np.array([1.2, -0.4, 0]))
    sc.play(FadeIn(cap), FadeIn(upd),
            GrowArrow(arrow(cap.get_right(), upd.get_left(), DIM, 0.08)), run_time=0.8)
    sc.play(GrowArrow(arrow(np.array([-2.0, 1.2, 0]), cap.get_top(), DIM, 0.08)),
            GrowArrow(arrow(upd.get_top(), m.get_bottom(), DIM, 0.1)), run_time=0.6)

    si = arrow(np.array([-4.6, -0.4, 0]), cap.get_left(), C_GRF, 0.08)
    sit = mono("scan_in", 16, C_GRF).next_to(si, LEFT, buff=0.1)
    so = arrow(cap.get_bottom(), np.array([-2.0, -1.6, 0]), C_GRF, 0.08)
    sot = mono("scan_out", 16, C_GRF).next_to(so, DOWN, buff=0.1)
    sc.play(GrowArrow(si), FadeIn(sit), GrowArrow(so), FadeIn(sot), run_time=0.6)

    exp = code_block([
        "capture_reg  is IN the scan chain. It samples the system value at Capture-DR,",
        "             then shifts - so shifting never disturbs what the cell drives.",
        "update_reg   holds what the cell drives. It only moves at Update-DR, so the",
        "             whole boundary changes at once, at a point the test controls.",
    ], 18, INK)
    exp.next_to(VGroup(cap, upd), DOWN, buff=1.0).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in exp], lag_ratio=0.2), run_time=1.8)
    sc.wait(1.4)

    sc.play(FadeOut(VGroup(dpin, dout, wire, m, cap, upd, si, sit, so, sot, exp)),
            run_time=0.6)

    modes = VGroup()
    for k, (n, c, w) in enumerate([
            ("SAMPLE", C_PY, "drives nothing, just watches - the fabric runs from the real switches"),
            ("EXTEST", C_VPR, "the boundary drives the PADS; the fabric is isolated"),
            ("INTEST", C_BIT, "the boundary drives the FABRIC's inputs and captures its outputs")]):
        modes.add(VGroup(mono(n, 24, c), Text(w, font_size=18, color=DIM))
                  .arrange(RIGHT, buff=0.4, aligned_edge=DOWN))
    modes.arrange(DOWN, aligned_edge=LEFT, buff=0.4).set_x(0).shift(UP * 0.7)
    if modes.width > 12.8:
        modes.scale_to_fit_width(12.8)
    sc.play(LaggedStart(*[FadeIn(m2, shift=RIGHT * 0.2) for m2 in modes], lag_ratio=0.25),
            run_time=1.6)

    size = code_block([
        "44 pads  x  2 cells  =  88 boundary cells",
        "cell k      = pad k's OUTPUT cell (fabric -> world, forced 0 while GTS)",
        "cell 44 + k = pad k's INPUT  cell (world -> fabric)",
    ], 19, C_GRF)
    size.next_to(modes, DOWN, buff=0.7).set_x(0)
    sc.play(FadeIn(size), run_time=0.9)

    gotcha = Text("Gotcha: a BC_1 cell captures its system input in EVERY mode - "
                  "so in INTEST the input cells read the real switches, not your vector.",
                  font_size=18, color=C_ERR)
    gotcha.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(gotcha), run_time=0.9)
    sc.wait(2.4)


def s7_files(sc):
    sc.files_used(
        inputs=[("hw/src/core/jtag_tap6.v", "state machine, 6-bit IR, DR select"),
                ("hw/src/core/bsc_cell.v", "one BC_1 boundary cell"),
                ("hw/src/fabric/bob_fpga.v", "wires the ring, 2 cells per pad")],
        generated=[("hw/src/generated/bob_params.vh", "BOB_BSR_W = 88, pad numbers"),
                   ("tools/bob/device.json", "bsr cells, board pad map")],
        verified=[("hw/tb/tb_bob.v", "IR, BYPASS, boundary, INTEST/EXTEST"),
                  ("host/dirtyjtag.py", "the same shifts on real hardware"),
                  ("docs/hwtest/*.md", "idcode + bypass open every board run")])


EP02 = [s1_why, s2_tap, s3_timing, s4_ir, s5_status, s6_boundary, s7_files]


class Ep02JTAG(BobScene):
    def construct(self):
        self.titlecard("EPISODE 2", "JTAG",
                       "the TAP, the instruction register, the boundary ring")
        for i, part in enumerate(EP02):
            part(self)
            if i < len(EP02) - 1:
                clear_all(self)


class E02S1Why(BobScene):
    def construct(self): s1_why(self)


class E02S2Tap(BobScene):
    def construct(self): s2_tap(self)


class E02S3Timing(BobScene):
    def construct(self): s3_timing(self)


class E02S4Ir(BobScene):
    def construct(self): s4_ir(self)


class E02S5Status(BobScene):
    def construct(self): s5_status(self)


class E02S6Boundary(BobScene):
    def construct(self): s6_boundary(self)


class E02S7Files(BobScene):
    def construct(self): s7_files(self)